In [ ]:
using Pkg
Pkg.activate("blast_code")
Pkg.resolve()
Pkg.instantiate()
include("blast_code/src/Blast.jl")
include("blast_code/src/blast_tutorials.jl")

In [ ]:
include("blast_code/src/new_funcs.jl")

In [ ]:
# Compact weighted window function + T_tilde for galaxy-galaxy (CC) probes
function compute_T̃_cc_weighted(ℓ, χ, R, Hz, nz, kmin, kmax; n_cheb=119, N=2^15+1)
    x, w = Blast.get_clencurt_grid(kmin, kmax, N), Blast.get_clencurt_weights(kmin, kmax, N)
    T, B_k = Blast.bessel_cheb_eval(ℓ, kmin, kmax, χ, n_cheb, N)
    W = @tullio W[i,k] := window_prefactor(Hz, nz)[i] * B_k[i,k]  # Weighted window: f(χ)·j_ℓ(kχ)
    
    T_CC = zeros(length(χ), length(R), n_cheb+1)
    α = w .* (x .^ 2)  # β=+2 for CC
    
    Threads.@threads for ridx in eachindex(R)
        B_k2 = @. SpecialFunctions.sphericalbesselj(ℓ, x * χ[1] / R[ridx])
        @tturbo for l in 1:n_cheb+1, i in 1:length(χ)
            s = zero(eltype(x))
            for k in 1:N
                s += T[l,k] * W[i,k] * B_k2[k] * α[k]
            end
            T_CC[i, ridx, l] = s
        end
    end
    return T_CC
end

In [ ]:
using NPZ
using DataInterpolations
using Interpolations
using FastChebInterp
using BenchmarkTools
using Plots
using QuadGK
using LaTeXStrings
using Tullio
using LoopVectorization
using LinearAlgebra
using Revise
using SpecialFunctions
using .Blast
using .blast_tutorials

## Step 1: Load N5K data

In [ ]:
#Background quantities
z_b = npzread("blast_code/data/background/z.npy")
χ_b = npzread("blast_code/data/background/chi.npy")
z_of_χ = DataInterpolations.AkimaInterpolation(z_b, χ_b);

In [ ]:
#3D matter power spectrum
pk_dict = npzread("blast_code/data/pk.npz")
Pklin = pk_dict["pk_lin"]
Pknonlin = pk_dict["pk_nl"]
k = pk_dict["k"]
z = pk_dict["z"];

In [ ]:
#Interpolating the power spectrum
#Linear P(k)
y = LinRange(log10(first(k)),log10(last(k)), length(k))
x = LinRange(first(z), last(z), length(z))

InterpPmm = Interpolations.interpolate(log10.(Pklin),BSpline(Cubic(Line(OnGrid()))))
InterpPmm = scale(InterpPmm, (x, y))
InterpPmm = Interpolations.extrapolate(InterpPmm, Line())

#Non-linear P(k)
y = LinRange(log10(first(k)),log10(last(k)), length(k))
x = LinRange(first(z), last(z), length(z))
InterpPmm_nl = Interpolations.interpolate(log10.(Pknonlin),BSpline(Cubic(Line(OnGrid()))))
InterpPmm_nl = scale(InterpPmm_nl, x, y)
InterpPmm_nl = Interpolations.extrapolate(InterpPmm_nl, Line())

#Callables
power_spectrum(k, χ1, χ2) = @. sqrt(10^InterpPmm(z_of_χ(χ1),log10(k)) * 10^InterpPmm(z_of_χ(χ2),log10(k)))
power_spectrum_nl(k, χ1, χ2) = @. sqrt(10^InterpPmm_nl(z_of_χ(χ1),log10(k)) * 10^InterpPmm_nl(z_of_χ(χ2),log10(k)));

In [ ]:
#N5K benchmarks
dtype = Float64

benchmark_gg = npzread("blast_code/data/benchmarks_nl_full_clgg.npz")
benchmark_ll = npzread("blast_code/data/benchmarks_nl_full_clss.npz")
benchmark_gl = npzread("blast_code/data/benchmarks_nl_full_clgs.npz")

#Extracting C_ℓ
gg = dtype.(benchmark_gg["cls"])
ll = dtype.(benchmark_ll["cls"])
gl = dtype.(benchmark_gl["cls"])
ell = dtype.(benchmark_gg["ls"])

#Reshaping them into the same Blast's format
gg_reshaped = zeros( dtype, length(ell), 10, 10)
counter = 1

for i in 1:10
    for j in i:10
        gg_reshaped[:,i,j] = gg[counter, :]
        gg_reshaped[:,j,i] = gg_reshaped[:,i,j]
        counter += 1
    end
end

ll_reshaped = zeros(dtype, length(ell), 5, 5)
counter = 1

for i in 1:5
    for j in i:5
        ll_reshaped[:,i,j] = ll[counter, :]
        ll_reshaped[:,j,i] = ll_reshaped[:,i,j]
        counter += 1
    end
end

gl_reshaped = zeros( dtype, length(ell), 10, 5)
counter = 1

for i in 1:10
    for j in 1:5
        gl_reshaped[:,i,j] = gl[counter, :]
        counter += 1
    end
end

## Using _Blast.jl_

When loading the module, some precomputed $\tilde{T}_\ell$ are already available, alongside the list of the corresponding $\ell$ values. $\tilde{T}_\ell$ are defined as:  $$\tilde{T}_{n ; \ell}^{\mathrm{AB}}\left(\chi_1, \chi_2\right) \equiv \int_{k_{\min }}^{k_{\max }} \mathrm{d} k f^{\mathrm{AB}}(k) T_n(k) j_{\ell}\left(k \chi_1\right) j_{\ell}\left(k\chi_2\right)$$
with:
$$f^{\mathrm{AB}}(k)= \begin{cases}k^2 & \mathrm{AB}=g g, \\ 1 / k^2 & \mathrm{AB}=s s, \\ 1 & \mathrm{AB}=g s.\end{cases}$$

In [ ]:
ℓ = Blast.ℓ
n_chi = 20
x_min = 26
x_max = 7000
χ = LinRange(x_min, x_max, n_chi)
R = chebpoints(n_chi, -1, 1)
R = reverse(R[R.>0])
nR = length(R)
kmax = 200/13
kmin = 2.5/x_max
n_cheb = 30
β = 2 #0 per CL e 2 per CC
k_cheb = chebpoints(n_cheb, log10(kmin), log10(kmax));
N = 2^(15)+1

In [ ]:
cosmo = Blast.FlatΛCDM()
z_range = z_of_χ.(χ)
bgrid = Blast.CosmologicalGrid(z_range = z_range)
bg = Blast.BackgroundQuantities(Hz_array = zeros(length(z_range)), χz_array = Array(χ));

In [ ]:
# Load N5K n(z), needed to compute lensing and clustering kernels
n5k_bins = npzread("blast_code/data/dNdzs_fullwidth.npz")
#Loading the N5K pre-computed kernel to check that they match!
W = npzread("blast_code/data/kernels_fullwidth.npz")

In [ ]:
GalKernels = Blast.GalaxyKernel(10, length(bgrid.z_range))
Blast.compute_kernel!(n5k_bins["dNdz_cl"]',  n5k_bins["z_cl"], GalKernels, bgrid, bg, cosmo)
ShearKernels = Blast.ShearKernel(4, length(bgrid.z_range))
Blast.compute_kernel!(n5k_bins["dNdz_sh"]', n5k_bins["z_sh"], ShearKernels, bgrid, bg, cosmo)

In [ ]:
# Define wavenumber grid for Hankel window transforms (can use k_cheb or define separately)
# Using the same k grid as the power spectrum Chebyshev points
k_window = 10 .^ k_cheb

# Extract clustering window from the computed kernels (normalized)
W_clustering = zeros(length(bgrid.z_range))
for i in 1:length(bgrid.z_range)
    W_clustering[i] = GalKernels.Kernel[1, i]  # Using first bin as example
end

# Normalize the window
W_clustering_normalized = W_clustering ./ maximum(W_clustering)

println("Window grid setup:")
println("  k_window length: ", length(k_window))
println("  χ grid length: ", length(χ))
println("  W_clustering length: ", length(W_clustering_normalized))

In [ ]:
# Compute Hankel-transformed window functions for clustering
# W̃_ℓ^{den}(k, χ₀) = ∫ dχ' χ'² W_g(χ') j_ℓ(k χ₀) j_ℓ(k χ')
Wtilde_CC_list = []
for (i, ell_val) in enumerate(ℓ)
    Wtilde_CC_single = computehankelwindowCC(round(Int, ell_val), χ, k_window, W_clustering_normalized)
    push!(Wtilde_CC_list, Wtilde_CC_single)
    if i % 5 == 0
        println("Computed Hankel windows for ℓ = $ell_val (index $i/$(length(ℓ)))")
    end
end

println("Hankel window computation complete. Shape of each window: ", size(Wtilde_CC_list[1]))

In [ ]:
cheb_coeff = zeros(n_chi, nR, n_cheb+1)
#Blast allows you to use FFT plans (in the FFTW fashion). In this way, the heaviest part of the computation is performed only once.
#plan = Blast.plan_fft(power_spectrum.(10 .^ k_cheb,χ[1],χ[1]*R[1]),1) 
plan = Blast.plan_fft(power_spectrum.(10 .^ k_cheb,χ[1],χ[1]*R[1])) 

for i in 1:nR 
    for j in 1:n_chi
        cheb_coeff[j,i,:] = Blast.fast_chebcoefs(power_spectrum.(10 .^ k_cheb,χ[j],χ[j]*R[i]), plan); 
    end
end

In [ ]:
# Example: compute w_CC directly using the full pipeline with Hankel windows
# This combines Hankel window application + T̃ computation + Chebyshev contraction
# into a single efficient operation

# First, ensure we have Chebyshev coefficients computed (from earlier cell)
# cheb_coeff shape: (n_chi, nR, n_cheb+1)

# For a single multipole as example:
example_ell = Int(ℓ[1])
w_CC_hankel_direct = computewCCwithwindows(example_ell, χ, R, kmin, kmax,
                                            Wtilde_CC_list[1], Wtilde_CC_list[1],
                                            k_window, cheb_coeff;
                                            n_cheb = n_cheb, N = N)

println("Direct w_CC computation with Hankel windows complete")
println("Shape of w_CC_hankel: ", size(w_CC_hankel_direct))
println("Max value: ", maximum(abs.(w_CC_hankel_direct)))
println("Min value: ", minimum(abs.(w_CC_hankel_direct)))

In [ ]:
# Compute T̃ for each ℓ value using Hankel-transformed windows
# Using computehankelwindowCC and compute_T̃_with_windows from new_funcs.jl
T_LL = zeros(length(ℓ), length(χ), length(R), n_cheb + 1)
T_CL = zeros(length(ℓ), length(χ), length(R), n_cheb + 1)
T_CC = zeros(length(ℓ), length(χ), length(R), n_cheb + 1)

for i in eachindex(ℓ)
    # For clustering (CC): use Hankel-transformed windows
    T_CC_with_hankel = compute_T̃_with_windows(ℓ[i], χ, R, kmin, kmax, 
                                                Wtilde_CC_list[i], Wtilde_CC_list[i], k_window;
                                                n_cheb = n_cheb, N = N)
    T_CC[i,:,:,:] = dropdims(T_CC_with_hankel; dims=1)
    
    # For now, keep LL and CL with standard computation
    # (These can be updated with Hankel transforms for shear and cross correlations if windows are available)
    T_LL[i,:,:,:] = Blast.compute_T̃(ℓ[i], χ, R, kmin, kmax, -2; n_cheb, N)
    T_CL[i,:,:,:] = Blast.compute_T̃(ℓ[i], χ, R, kmin, kmax, 0; n_cheb, N)
    
    println("done ℓ[$i] = $(ℓ[i]), now ℓ[$(i+1)]") 
end

In [ ]:
print("Shape of the precomputed T̃_LL: ", size(T_LL), "\n")
print("Shape of the precomputed T̃_CL: ", size(T_CL), "\n")
print("Shape of the precomputed T̃_CC: ", size(T_CC), "\n")

#### Using `computewCCwithwindows` for direct pipeline

Alternative: Use the high-level pipeline `computewCCwithwindows` which combines Hankel window computation with T̃ contraction and Chebyshev decomposition in a single call. This is more efficient when you want to compute w_CC directly.

#### Workflow Summary: Hankel Transform Integration

The modified pipeline now includes realistic window functions via Hankel transforms:

1. **Window Setup**: Define galaxy clustering window $W_g(\chi)$ from computed kernels
2. **Hankel Transform**: Compute $\tilde{W}(k, \chi_0) = \int d\chi' \chi'^2 W_g(\chi') j_\ell(k\chi_0) j_\ell(k\chi')$
3. **T̃ Computation**: Build BLAST coefficients $\tilde{T}_{n;\ell}^{CC}(\chi_1, \chi_2)$ with window contributions  
4. **Contraction**: Combine with Chebyshev coefficients to get $w_{CC}(\chi_1, R)$

**Two available workflows**:
- **T̃-centered** (cell with T_LL/T_CL/T_CC): Precompute all T̃ tensors once, reuse for integration kernels
- **Direct pipeline** (computewCCwithwindows): Single-step computation for immediate w_CC without intermediate storage

In [ ]:
w_LL = Blast.w_ell_tullio(cheb_coeff, T_LL)
w_CL = Blast.w_ell_tullio(cheb_coeff, T_CL)
w_CC = Blast.w_ell_tullio(cheb_coeff, T_CC);

In [ ]:
@benchmark Blast.w_ell_tullio(cheb_coeff, T_LL)

In [ ]:
heatmap(1:nR, χ, w_LL[1,:,:]./maximum(w_LL[1,:,:]), size = (500,500), title=L"w_{LL}(ℓ=2.0)", 
    c =:roma , xlabel=L"R=\chi_1/\chi_2",ylabel=L"\chi[Mpc/h]", legend=:none,
    yguidefontsize=15, xguidefontsize=15 , titlefontsize=20)

# plot!([16,16],[26, 7200],arrow=true,color=:black,linewidth=3,label=nothing, size = (500,500)) 
# annotate!(11, 7300, text(L"R=0.5",14), :black)
# #annotate!(16, -200,text(L"0.5",10), :black)

# plot!([28,28],[26, 7200],arrow=true,color=:black,linewidth=3,label=nothing, size = (500,500)) 
# annotate!(23, 7300, text(L"R=0.8",14), :black)
# #annotate!(28, -200, text(L"0.8",10), :black)

# plot!([34,34],[26, 7200],arrow=true,color=:black,linewidth=3,label=nothing,axis=([], false), size = (500,500)) 
# annotate!(34, 7400, text(L"R=0.9",14), :black)

In [ ]:
heatmap(1:nR, χ, w_LL[22,:,:]./maximum(w_LL[22,:,:]), size = (500,500), title=L"w_{LL}(ℓ=211.63)",
    legend = :none, c =:roma , xlabel=L"R=\chi_1/\chi_2",ylabel=L"\chi[Mpc/h]",
    yguidefontsize=15, xguidefontsize=15 , titlefontsize=20)

# plot!([16,16],[26, 7200],arrow=true,color=:black,linewidth=3,label="R=0.5", size = (500,500)) 
# annotate!(11, 7300, text(L"R=0.5",14), :black)
# #annotate!(16, -200,text(L"0.5",10), :black)

# plot!([28,28],[26, 7200],arrow=true,color=:black,linewidth=3,label="R=0.5", size = (500,500)) 
# annotate!(23, 7300, text(L"R=0.8",14), :black)
# #annotate!(28, -200, text(L"0.8",10), :black)

# plot!([34,34],[26, 7200],arrow=true,color=:black,linewidth=3,label="R=0.5",axis=([], false), size = (500,500)) 
# annotate!(34, 7400, text(L"R=0.9",14), :black)

## Computing the integration kernels

#### Step 1: defining background quantities 
This is not showing the optimal usage of Blast, but how to make the $C_\ell$'s match to the N5K challenge benchmarks using the precomputed quantites as showed in the paper (https://arxiv.org/abs/2410.03632). 

In [ ]:
i_bin = 1

interp = DataInterpolations.AkimaInterpolation( W["kernels_cl"][i_bin,:], W["z_cl"], extrapolation = ExtrapolationType.Linear)
intn, _ = quadgk(x -> interp.(x), 0., 3.5)

interp = DataInterpolations.AkimaInterpolation( GalKernels.Kernel[i_bin,:], bgrid.z_range, extrapolation = ExtrapolationType.Linear)
intb, _ = quadgk(x -> interp.(x), 0., 3.5)

plot(bgrid.z_range, GalKernels.Kernel[i_bin,:]/intb, label="BLAST", title = "Clustering kernels")
plot!(W["z_cl"], W["kernels_cl"][i_bin,:]/intn, label="N5K")

In [ ]:
bin = 4

plot(bgrid.z_range, ShearKernels.Kernel[bin,:], label="BLAST", title = "Shear kernels")
plot!(W["z_sh"], W["kernels_sh"][bin,:], label="N5K")

#### Finally, compute the Cℓ

The module blast_tutorial contains the functions that were specifically developed to work with the N5K inputs and were not included in the official module.

In [ ]:
K_CC, K_CL, K_LL = blast_tutorials.compute_kernels(W, χ, R)
w_χ = Blast.simpson_weight_array(n_chi)
w_R = Blast.get_clencurt_weights_R_integration(2*nR+1)
pref_CC = Blast.get_ell_prefactor(GalKernels, GalKernels, ℓ)
pref_CL = Blast.get_ell_prefactor(ShearKernels, GalKernels, ℓ)
pref_LL = Blast.get_ell_prefactor(ShearKernels, ShearKernels, ℓ);

In [ ]:
clustering_Cℓ = Blast.compute_Cℓ(w_CC, K_CC, bg, w_χ, w_R, pref_CC);
#shear_Cℓ = Blast.compute_Cℓ(w_LL, K_LL, bg, w_χ, w_R, pref_LL)
#cross_Cℓ = Blast.compute_Cℓ(w_CL, K_CL, bg, w_χ, w_R, pref_CL);

In [ ]:
@benchmark Blast.compute_Cℓ(w_CC, K_CC, bg, w_χ, w_R, pref_CC)

To match the N5K $C_\ell$'s, we treat the linear, $P_{\mathrm{lin}}(k)$, and non-linear, $P_{\delta}(k)$, matter power spectrum as two separate components and perform the splitting: $$P_\delta\left(k, \chi_1, \chi_2\right)=P_{\operatorname{lin}}\left(k, \chi_1, \chi_2\right)+\left[P_\delta-P_{\operatorname{lin}}\right]\left(k, \chi_1, \chi_2\right)$$
For the linear component, we performed the Chebyshev decomposition as defined before and used that approximation to evaluate the non-Limber angular power spectrum. The non-linear part, on the other hand, is only relevant on small scales, where the Limber approximationis sufficiently accurate. In the following cell, I am loading the $C_\ell$'s computed in the same $\ell$ points, but using the Limber approximation. The functions used to obtain them are included in the blast_tutorial module for completeness.

In [ ]:
Cℓ_CC_limb = dtype.(npzread("blast_code/data/Limber/Cl_CC_limber_linear_full.npy"));
Cℓ_CC_limb_nl = dtype.(npzread("blast_code/data/Limber/Cl_CC_limber_nl_full.npy"));

In [ ]:
final_clustering_Cℓ = clustering_Cℓ + Cℓ_CC_limb_nl - Cℓ_CC_limb;

In the challenge, the $C_\ell$'s are evaluated in the range $2<\ell<2000$. After $\ell = 200$, the Limber approximation is accurate enough. We evaluated the $C_\ell$'s in a set of $\ell$ points which are $100$ Chebyshev points defined in the interval of interest. With this choice, we are able to interpolate using, once again, the Chebyshev polynomials and put our angular power spectra on the same N5K $\ell$ grid.

In [ ]:
### add Limber Cl's in chebyshev points for l>200
Cℓ_CC_limb = dtype.(npzread("blast_code/data/Limber/Cl_CC_limber+200_full.npy"));
total_Cℓ_CC = cat(final_clustering_Cℓ, Cℓ_CC_limb, dims=1);
print(total_Cℓ_CC)

In [ ]:
# Interpolating to go on the same n5k grid
elle = dtype.(reverse(chebpoints(100, 2, 2000)))
ℓ_min = 2
ℓ_max = 2000

interp_Cℓ_CC = zeros(dtype, length(ell), 10, 10)
interp_Cℓ_CL = zeros(dtype, length(ell), 10, 5)
interp_Cℓ_LL = zeros(dtype, length(ell), 5, 5)

for i in 1:10
    for j in i:10
        interpol = chebinterp(reverse(total_Cℓ_CC[:,i,j].*elle.*elle), ℓ_min, ℓ_max)
        interp_Cℓ_CC[:,i,j] = interpol.(ell) ./ (ell.*ell)
        interp_Cℓ_CC[:,j,i] = interp_Cℓ_CC[:,i,j]
    end
end

In [ ]:
i = 10
j = 10
nl = 60 #set this to 103 to see up to ℓ=2000.
plot(ell[1:nl], interp_Cℓ_CC[1:nl,i,j], label="BLAST", title = L"$C_\ell^{gg}$", titlefontsize=20, 
    xlabel=L"$\ell$", ylabel=L"$C_\ell$", labelfontsize=15)
plot!(ell[1:nl], gg_reshaped[1:nl,i,j], label = "N5K")

In [ ]:
i = 10
j = 10
nl = 103 #set this to 103 to see up to ℓ=2000.
plot(ell[1:nl], interp_Cℓ_CC[1:nl,i,j] .* ell[1:nl] .* (ell[1:nl] .+1), label="BLAST", title = L"$C_\ell^{gg}$", titlefontsize=20, 
    xlabel=L"$\ell$", ylabel=L"$\ell(\ell+1)C_\ell$", labelfontsize=15)
plot!(ell[1:nl], gg_reshaped[1:nl,i,j] .* ell[1:nl] .* (ell[1:nl] .+1), label = "N5K")